#### Even Time VS Processing Time 
- Always perform aggregation on Even Time instead of Processing Time, as there are chances that data can get delayed from source to delta lake or whatever input place where it is brought from actual source of data.

Aggregation in Window means:
- You want to find how many times (agg) something happened for which you want aggregation for given window or time frame like 10 seconds etc.
- Aggregation based on key column, and window

#### Tumbling Windows:
- Its a Non-Overlapping one
- Fixed Sized Window.

In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
CREATE TABLE dbcertification.streaming.windowtbl
(
  color STRING,
  event_date TIMESTAMP
)
location 'abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/windowtbl'

In [0]:
%sql
INSERT INTO dbcertification.streaming.windowtbl
VALUES
('red','2025-01-01T11:01:00.000+00:00'),
('green','2025-01-01T11:01:00.000+00:00')

In [0]:
df = spark.readStream.table("dbcertification.streaming.windowtbl")

In [0]:
df = df.groupBy("color",window("event_date","10 minutes")) \
        .agg(count(lit(1)).alias("color_count"))

In [0]:
df.writeStream.format("delta")\
        .outputMode("Complete")\
        .trigger(once=True)\
        .option("path","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/windowtblsink/")\
        .option("checkpointLocation","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/checkpointLocation")\
        .start()

In [0]:
%sql
select * from delta.`abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/windowtblsink/`

In [0]:
%sql
INSERT INTO dbcertification.streaming.windowtbl
VALUES
('green','2025-01-01T11:07:00.000+00:00')

In [0]:
df.writeStream.format("delta")\
        .outputMode("Complete")\
        .trigger(once=True)\
        .option("path","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/windowtblsink/")\
        .option("checkpointLocation","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/checkpointLocation")\
        .start()

In [0]:
%sql
select * from delta.`abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/windowtblsink/`

In [0]:
%sql
INSERT INTO dbcertification.streaming.windowtbl
VALUES
('green','2025-01-01T11:12:00.000+00:00')

In [0]:
df.writeStream.format("delta")\
        .outputMode("Complete")\
        .trigger(once=True)\
        .option("path","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/windowtblsink/")\
        .option("checkpointLocation","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/checkpointLocation")\
        .start()

In [0]:
%sql
select * from delta.`abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/windows/windowtblsink/`

#### Window Types
- Sliding Window 
- Session Window

#### WATERMARK

In [0]:
%sql
INSERT INTO dbcertification.streaming.windowtbl
VALUES
('red','2025-01-01T11:07:00.000+00:00')